# Step 4 — Build ML Pipeline with DVC

En este notebook se valida que el proyecto ya puede ejecutarse como un pipeline reproducible usando DVC.

El flujo deja de depender de ejecutar scripts manualmente y pasa a correr con:

dvc repro

El pipeline contiene cuatro stages:

1. prepare_data
2. split_data
3. train
4. evaluate

El objetivo es confirmar que DVC reproduce las mismas métricas obtenidas en el monolito de Fase 1.

# Imports

In [3]:
# IMPORTS

from pathlib import Path
import subprocess
import json
import yaml
import pandas as pd

# Config

In [5]:
# CONFIG

PROJECT_ROOT = Path.cwd()

PARAMS_PATH = PROJECT_ROOT / "params.yaml"
DVC_YAML_PATH = PROJECT_ROOT / "dvc.yaml"
DVC_LOCK_PATH = PROJECT_ROOT / "dvc.lock"

with open(PARAMS_PATH, "r", encoding="utf-8") as f:
    params = yaml.safe_load(f)

print("Project root:", PROJECT_ROOT)
print("params.yaml exists:", PARAMS_PATH.exists())
print("dvc.yaml exists:", DVC_YAML_PATH.exists())
print("dvc.lock exists:", DVC_LOCK_PATH.exists())

params

Project root: C:\Users\daniel.martinez\real_state_price_predictor\tog_dme
params.yaml exists: True
dvc.yaml exists: True
dvc.lock exists: True


{'base': {'numpy_seed': 33},
 'data_load': {'raw_data_path': 'data/raw/Guadalajara 4Q22.xlsx'},
 'prepare': {'prepared_data_path': 'data/processed/prepared_data.csv',
  'delivery_base_date': '2022-09-01',
  'towns_to_drop': ['Chapala', 'El Salto'],
  'price_per_sqm_max': 90000,
  'sqm_max': 250.0,
  'classification_mapping': {'S': -2, 'E': -1, 'M': 0, 'R': 1, 'RP': 2},
  'dead_columns': ['id',
   'project_name',
   'address',
   'promoter',
   'alcoba',
   'room_serv',
   'first_price',
   'initial_date',
   'entry_date',
   'update_date',
   'fin_op',
   'inventory_months'],
  'final_drop_columns': ['colony',
   'town',
   'price_per_sqm',
   'comm_succ',
   'absortion',
   'sold_units',
   'baths']},
 'split': {'test_size': 0.2,
  'random_state': 42,
  'train_data_path': 'data/processed/train_data.csv',
  'test_data_path': 'data/processed/test_data.csv'},
 'train': {'model_type': 'ridge',
  'scoring': 'r2',
  'cv_splits': 10,
  'cv_random_state': 42,
  'n_jobs': -1,
  'alphas': [1000

## 1. Helper para comandos

In [7]:
# HELPER FUNCTION

def run_command(command):
    """
    Ejecuta comandos de terminal desde el notebook.
    Usa shell=True porque estamos trabajando en Windows.
    """
    result = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        shell=True
    )

    if result.stdout:
        print(result.stdout)

    if result.stderr:
        print("STDERR:")
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"Falló el comando: {command}")

    return result

## 1. Validar archivos principales

In [9]:
# CHECK REQUIRED FILES

required_files = [
    "params.yaml",
    "dvc.yaml",
    "src/stages/prepare_data.py",
    "src/stages/split_data.py",
    "src/stages/train.py",
    "src/stages/evaluate.py",
]

for file_path in required_files:
    path = PROJECT_ROOT / file_path
    print(file_path, "->", path.exists())

    if not path.exists():
        raise FileNotFoundError(f"No existe archivo requerido: {file_path}")

params.yaml -> True
dvc.yaml -> True
src/stages/prepare_data.py -> True
src/stages/split_data.py -> True
src/stages/train.py -> True
src/stages/evaluate.py -> True


## 2. Revisar estado inicial de DVC

In [11]:
# DVC STATUS BEFORE REPRO

run_command("dvc status")

Data and pipelines are up to date.



CompletedProcess(args='dvc status', returncode=0, stdout='Data and pipelines are up to date.\n', stderr='')

## 3. Ejecutar pipeline DVC

In [13]:
# RUN DVC REPRO

run_command("dvc repro")

Stage 'prepare_data' didn't change, skipping
Stage 'split_data' didn't change, skipping
Stage 'train' didn't change, skipping
Stage 'evaluate' didn't change, skipping
Data and pipelines are up to date.



CompletedProcess(args='dvc repro', returncode=0, stdout="Stage 'prepare_data' didn't change, skipping\nStage 'split_data' didn't change, skipping\nStage 'train' didn't change, skipping\nStage 'evaluate' didn't change, skipping\nData and pipelines are up to date.\n", stderr='')

## 4. Revisar estado final de DVC

In [15]:
# DVC STATUS AFTER REPRO

run_command("dvc status")

Data and pipelines are up to date.



CompletedProcess(args='dvc status', returncode=0, stdout='Data and pipelines are up to date.\n', stderr='')

## 5. Visualizar DAG del pipeline

In [17]:
# DVC DAG

run_command("dvc dag")

       +--------------+    
       | prepare_data |    
       +--------------+    
               *           
               *           
               *           
        +------------+     
        | split_data |     
        +------------+     
         **        **      
       **            **    
      *                **  
+-------+                * 
| train |              **  
+-------+            **    
         **        **      
           **    **        
             *  *          
         +----------+      
         | evaluate |      
         +----------+      



CompletedProcess(args='dvc dag', returncode=0, stdout='       +--------------+    \n       | prepare_data |    \n       +--------------+    \n               *           \n               *           \n               *           \n        +------------+     \n        | split_data |     \n        +------------+     \n         **        **      \n       **            **    \n      *                **  \n+-------+                * \n| train |              **  \n+-------+            **    \n         **        **      \n           **    **        \n             *  *          \n         +----------+      \n         | evaluate |      \n         +----------+      \n', stderr='')

## 6. Mostrar métricas con DVC

In [19]:
# DVC METRICS SHOW

run_command("dvc metrics show")

Path                  MAE_test_pesos    MAE_train_pesos    R2_test    R2_train    RMSE_test_pesos    RMSE_train_pesos
reports\metrics.json  549476.60932      598614.59216       0.85867    0.87228     724228.72872       820764.7608



CompletedProcess(args='dvc metrics show', returncode=0, stdout='Path                  MAE_test_pesos    MAE_train_pesos    R2_test    R2_train    RMSE_test_pesos    RMSE_train_pesos\nreports\\metrics.json  549476.60932      598614.59216       0.85867    0.87228     724228.72872       820764.7608\n', stderr='')

## 7. Validar outputs generados

In [21]:
# VALIDATE OUTPUT FILES

expected_outputs = [
    params["prepare"]["prepared_data_path"],
    params["split"]["train_data_path"],
    params["split"]["test_data_path"],
    params["artifacts"]["model_path"],
    params["artifacts"]["scaler_x_path"],
    params["artifacts"]["scaler_y_path"],
    params["artifacts"]["feature_names_path"],
    params["metrics"]["metrics_path"],
    "dvc.lock",
]

for output in expected_outputs:
    path = PROJECT_ROOT / output
    print(output, "->", path.exists())

    if not path.exists():
        raise FileNotFoundError(f"No se generó el output esperado: {output}")

data/processed/prepared_data.csv -> True
data/processed/train_data.csv -> True
data/processed/test_data.csv -> True
models/modelo_final.pkl -> True
models/scaler_X.pkl -> True
models/scaler_Y.pkl -> True
models/feature_names.json -> True
reports/metrics.json -> True
dvc.lock -> True


## 8. Revisar datasets generados

In [23]:
# CHECK GENERATED DATASETS

prepared_data = pd.read_csv(params["prepare"]["prepared_data_path"])
train_data = pd.read_csv(params["split"]["train_data_path"])
test_data = pd.read_csv(params["split"]["test_data_path"])

print("Prepared data shape:", prepared_data.shape)
print("Train data shape:", train_data.shape)
print("Test data shape:", test_data.shape)

prepared_data.head()

Prepared data shape: (536, 18)
Train data shape: (428, 18)
Test data shape: (108, 18)


,classification,sqm,terrace,bhk,park_u,levels,price,months_in_sale,master_plan_units,total_units,inventory,months_to_delivery,Guadalajara,Jocotepec,Tlajomulco de Zúñiga,Tlaquepaque,Tonalá,Zapopan
0,1,222.0,0.0,3.0,2.0,11.0,9077955.0,85.150685,81,44,4,0.0,True,False,False,False,False,False
1,1,111.0,0.0,2.0,2.0,11.0,5922418.0,85.150685,81,44,4,0.0,True,False,False,False,False,False
2,1,110.0,0.0,2.0,2.0,11.0,5867088.0,85.150685,81,44,4,0.0,True,False,False,False,False,False
3,1,91.0,0.0,2.0,2.0,10.0,5300000.0,84.295890,392,392,8,0.0,False,False,False,False,False,True
4,1,131.0,0.0,3.0,2.0,10.0,6503000.0,84.295890,392,392,8,0.0,False,False,False,False,False,True


## 9. Validar métricas finales

In [25]:
# LOAD METRICS

metrics_path = PROJECT_ROOT / params["metrics"]["metrics_path"]

with open(metrics_path, "r", encoding="utf-8") as f:
    metrics = json.load(f)

metrics

{'R2_train': 0.8722779520936734,
 'R2_test': 0.8586681107871221,
 'RMSE_train_pesos': 820764.7607973475,
 'RMSE_test_pesos': 724228.728716661,
 'MAE_train_pesos': 598614.5921648004,
 'MAE_test_pesos': 549476.6093178215}

## 10. Comparar métricas finales contra fase 1

In [27]:
# COMPARE AGAINST PHASE 1 RESULTS

expected_metrics = {
    "R2_test": 0.8586681107871221,
    "RMSE_test_pesos": 724228.7287166608,
    "MAE_test_pesos": 549476.6093178215,
}

print("Comparación contra monolito de Fase 1:")
print("-" * 60)

for metric_name, expected_value in expected_metrics.items():
    current_value = metrics[metric_name]
    difference = abs(current_value - expected_value)

    print(f"{metric_name}")
    print(f"  Esperado: {expected_value}")
    print(f"  Actual:   {current_value}")
    print(f"  Dif:      {difference}")
    print()

assert abs(metrics["R2_test"] - expected_metrics["R2_test"]) < 1e-8
assert abs(metrics["RMSE_test_pesos"] - expected_metrics["RMSE_test_pesos"]) < 1e-2
assert abs(metrics["MAE_test_pesos"] - expected_metrics["MAE_test_pesos"]) < 1e-2

print("Métricas validadas correctamente contra Fase 1.")

Comparación contra monolito de Fase 1:
------------------------------------------------------------
R2_test
  Esperado: 0.8586681107871221
  Actual:   0.8586681107871221
  Dif:      0.0

RMSE_test_pesos
  Esperado: 724228.7287166608
  Actual:   724228.728716661
  Dif:      2.3283064365386963e-10

MAE_test_pesos
  Esperado: 549476.6093178215
  Actual:   549476.6093178215
  Dif:      0.0

Métricas validadas correctamente contra Fase 1.


## Conclusión

El pipeline DVC reproduce correctamente el flujo completo del modelo.

Con este paso, el proyecto deja de depender de ejecutar scripts manualmente y puede reproducirse mediante:

dvc repro

Esto confirma que el modelo ganador de Fase 1 fue industrializado como pipeline reproducible.ducible.